Notebook: 03_delineation_validation.ipynb

Purpose: Quantify boundary uncertainty using available annotations and raw ECG waveforms.

Inputs:
- raw ECG waveforms
- delineation annotations

Outputs:
- delineation_features.parquet

# 03 — Delineation Validation

Estimate lead-level uncertainty for waveform fiducial boundaries and derive a boundary confidence score.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.datasets import PTBXLDataset, LUDBDataset, NSTDBDataset, INCARTDataset, QTDBDataset

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)
data_dir = root_dir / 'data'

adapters = {
    'ptbxl': PTBXLDataset,
    'ludb': LUDBDataset,
    'nstdb': NSTDBDataset,
    'incart': INCARTDataset,
    'qtdb': QTDBDataset,
}

rows = []

for dataset_name, adapter_cls in adapters.items():
    dataset_path = data_dir / dataset_name
    if not dataset_path.exists():
        continue
    adapter = adapter_cls(data_dir=data_dir)
    try:
        record_names = adapter.list_records()
    except Exception:
        continue

    for record_name in record_names:
        try:
            record = adapter.load_record(record_name)
        except Exception:
            continue

        record_id = f'{dataset_name}/{record_name}'
        signal = np.asarray(record.signal, dtype=float)
        if signal.ndim == 1:
            signal = signal.reshape(-1, 1)
        annotations = getattr(record, 'annotations', []) or []
        lead_names = getattr(record, 'lead_names', []) or []

        for lead_index, lead_name in enumerate(lead_names):
            lead_signal = signal[:, lead_index] if signal.shape[1] > lead_index else signal[:, 0]
            lead_annotations = [ann for ann in annotations if getattr(ann, 'lead', None) == lead_index]
            samples = [getattr(ann, 'sample', None) for ann in lead_annotations if getattr(ann, 'sample', None) is not None]
            window = int(max(1, 0.04 * float(getattr(record, 'fs', np.nan))))
            def sample_uncertainty(sample_list):
                values = []
                for sample in sample_list:
                    if sample is None or sample < 0 or sample >= lead_signal.size:
                        continue
                    left = max(0, sample - window)
                    right = min(lead_signal.size, sample + window)
                    segment = lead_signal[left:right]
                    if segment.size < 2:
                        continue
                    values.append(float(np.mean(np.abs(np.diff(segment)))))
                if not values:
                    return np.nan
                return float(np.std(values) / (np.mean(values) + 1e-12) * 1000.0)

            p_onsets = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() in {'(', 'p'}]
            p_offsets = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == ')' and getattr(getattr(ann, 'label', None), 'lower', lambda: '')().startswith('p')]
            qrs_onsets = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == 'n']
            qrs_offsets = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == ')']
            t_onsets = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() in {'(', 't'} and getattr(getattr(ann, 'label', None), 'lower', lambda: '')().startswith('t')]
            t_ends = [ann.sample for ann in lead_annotations if getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == ')']

            p_onset_unc = sample_uncertainty(p_onsets)
            p_offset_unc = sample_uncertainty(p_offsets)
            qrs_onset_unc = sample_uncertainty(qrs_onsets)
            qrs_offset_unc = sample_uncertainty(qrs_offsets)
            t_onset_unc = sample_uncertainty(t_onsets)
            t_end_unc = sample_uncertainty(t_ends)
            boundary_confidence = float(np.clip(1.0 - np.nanmean([p_onset_unc, p_offset_unc, qrs_onset_unc, qrs_offset_unc, t_onset_unc, t_end_unc]) / 100.0, 0.0, 1.0))

            rows.append({
                'record_id': record_id,
                'lead_id': str(lead_name).lower(),
                'p_onset_uncertainty_ms': p_onset_unc,
                'p_offset_uncertainty_ms': p_offset_unc,
                'qrs_onset_uncertainty_ms': qrs_onset_unc,
                'qrs_offset_uncertainty_ms': qrs_offset_unc,
                't_onset_uncertainty_ms': t_onset_unc,
                't_end_uncertainty_ms': t_end_unc,
                'boundary_confidence': boundary_confidence,
            })

if rows:
    delineation = pd.DataFrame(rows)
else:
    delineation = pd.DataFrame(columns=[
        'record_id','lead_id','p_onset_uncertainty_ms','p_offset_uncertainty_ms',
        'qrs_onset_uncertainty_ms','qrs_offset_uncertainty_ms','t_onset_uncertainty_ms',
        't_end_uncertainty_ms','boundary_confidence',
    ])

expected = {'record_id','lead_id','p_onset_uncertainty_ms','p_offset_uncertainty_ms','qrs_onset_uncertainty_ms','qrs_offset_uncertainty_ms','t_onset_uncertainty_ms','t_end_uncertainty_ms','boundary_confidence'}
assert expected.issubset(set(delineation.columns)), 'delineation feature schema mismatch'
assert delineation[['record_id','lead_id']].duplicated().sum() == 0

delineation.to_parquet(artifacts_dir / 'delineation_features.parquet', index=False)
print('Wrote delineation_features.parquet')
